In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# ML Models
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Tuning & Evaluation
from sklearn.model_selection import RandomizedSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data split dari preprocessing
X_train, X_test, y_train, y_test = joblib.load('models/data_split_regression.pkl')
feature_cols = joblib.load('models/feature_cols.pkl')

print("✅ Semua library berhasil diimport!")
print(f"\nShape X_train : {X_train.shape}")
print(f"Shape X_test  : {X_test.shape}")
print(f"Shape y_train : {y_train.shape}")
print(f"Shape y_test  : {y_test.shape}")
print(f"\nRange y_train : {y_train.min():.2f} - {y_train.max():.2f}")

✅ Semua library berhasil diimport!

Shape X_train : (3946, 12)
Shape X_test  : (987, 12)
Shape y_train : (3946,)
Shape y_test  : (987,)

Range y_train : 8.20 - 9.90


In [2]:
# Setup 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Fungsi evaluasi model
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)
    
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    mae_train  = mean_absolute_error(y_train, y_pred_train)
    r2_train   = r2_score(y_train, y_pred_train)
    
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_test  = mean_absolute_error(y_test, y_pred_test)
    r2_test   = r2_score(y_test, y_pred_test)
    
    r2_gap = r2_train - r2_test
    overfit_status = f"⚠️ OVERFIT (gap={r2_gap:.4f})" if r2_gap > 0.1 else f"✅ Normal (gap={r2_gap:.4f})"
    
    results = {
        'Model'      : model_name,
        'RMSE_Train' : round(rmse_train, 4),
        'RMSE_Test'  : round(rmse_test, 4),
        'MAE_Train'  : round(mae_train, 4),
        'MAE_Test'   : round(mae_test, 4),
        'R2_Train'   : round(r2_train, 4),
        'R2_Test'    : round(r2_test, 4),
        'Overfit'    : overfit_status
    }
    
    print(f"\n{'='*50}")
    print(f"📊 {model_name}")
    print(f"{'='*50}")
    print(f"{'Metrik':10s} {'Train':>10s} {'Test':>10s}")
    print(f"{'-'*32}")
    print(f"{'RMSE':10s} {rmse_train:>10.4f} {rmse_test:>10.4f}")
    print(f"{'MAE':10s} {mae_train:>10.4f} {mae_test:>10.4f}")
    print(f"{'R²':10s} {r2_train:>10.4f} {r2_test:>10.4f}")
    print(f"\nOverfitting Check: {overfit_status}")
    
    return results, y_pred_test

all_results = {}
all_predictions = {}
print("✅ Setup evaluasi selesai!")

✅ Setup evaluasi selesai!


Random Forest

In [3]:
# ── RANDOM FOREST ──────────────────────────────────────────
# Sebagai baseline tree-based model
# Tidak di-tune karena RF sudah robust dengan default parameter

print("🌲 Training Random Forest (Baseline)...")

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1  # pakai semua core CPU
)

rf_model.fit(X_train, y_train)

rf_results, rf_pred = evaluate_model(
    rf_model, X_train, X_test, y_train, y_test, 
    "Random Forest (Baseline)"
)

all_results['Random Forest'] = rf_results
all_predictions['Random Forest'] = rf_pred

joblib.dump(rf_model, 'models/rf_model.pkl')
print("\n✅ Model tersimpan: models/rf_model.pkl")

🌲 Training Random Forest (Baseline)...

📊 Random Forest (Baseline)
Metrik          Train       Test
--------------------------------
RMSE           0.0626     0.1613
MAE            0.0489     0.1290
R²             0.9321     0.5849

Overfitting Check: ⚠️ OVERFIT (gap=0.3472)

✅ Model tersimpan: models/rf_model.pkl


XGBoost


In [4]:
# ── XGBOOST ────────────────────────────────────────────────
# Model utama, di-tune dengan RandomizedSearchCV

print("⚡ Tuning XGBoost dengan RandomizedSearchCV...")
print("   (Estimasi waktu: 2-5 menit)")

xgb_param_dist = {
    'n_estimators'     : [100, 200, 300, 400, 500],
    'max_depth'        : [3, 4, 5, 6, 7, 8],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight' : [1, 3, 5, 7],
    'reg_alpha'        : [0, 0.1, 0.5, 1.0],
    'reg_lambda'       : [0.5, 1.0, 1.5, 2.0]
}

xgb_base = XGBRegressor(random_state=42, verbosity=0)

xgb_search = RandomizedSearchCV(
    estimator  = xgb_base,
    param_distributions = xgb_param_dist,
    n_iter     = 50,       # coba 50 kombinasi random
    cv         = kf,       # 5-Fold CV
    scoring    = 'r2',
    n_jobs     = -1,
    random_state = 42,
    verbose    = 1
)

xgb_search.fit(X_train, y_train)
xgb_model = xgb_search.best_estimator_

print(f"\n🏆 Best Parameters XGBoost:")
for param, val in xgb_search.best_params_.items():
    print(f"   {param:20s}: {val}")

xgb_results, xgb_pred = evaluate_model(
    xgb_model, X_train, X_test, y_train, y_test,
    "XGBoost (Tuned)"
)

all_results['XGBoost'] = xgb_results
all_predictions['XGBoost'] = xgb_pred

joblib.dump(xgb_model, 'models/xgb_model.pkl')
print("\n✅ Model tersimpan: models/xgb_model.pkl")

⚡ Tuning XGBoost dengan RandomizedSearchCV...
   (Estimasi waktu: 2-5 menit)
Fitting 5 folds for each of 50 candidates, totalling 250 fits

🏆 Best Parameters XGBoost:
   subsample           : 0.6
   reg_lambda          : 2.0
   reg_alpha           : 0.5
   n_estimators        : 200
   min_child_weight    : 3
   max_depth           : 4
   learning_rate       : 0.05
   colsample_bytree    : 0.7

📊 XGBoost (Tuned)
Metrik          Train       Test
--------------------------------
RMSE           0.1408     0.1556
MAE            0.1115     0.1248
R²             0.6569     0.6135

Overfitting Check: ✅ Normal (gap=0.0434)

✅ Model tersimpan: models/xgb_model.pkl


LightGBM

In [5]:
# ── LIGHTGBM ───────────────────────────────────────────────
# Comparator, di-tune dengan RandomizedSearchCV

print("🚀 Tuning LightGBM dengan RandomizedSearchCV...")
print("   (Estimasi waktu: 2-5 menit)")

lgbm_param_dist = {
    'n_estimators'      : [100, 200, 300, 400, 500],
    'num_leaves'        : [20, 31, 40, 50, 60, 80],   # parameter terpenting LightGBM
    'max_depth'         : [-1, 5, 7, 10, 15],
    'learning_rate'     : [0.01, 0.05, 0.1, 0.15, 0.2],
    'min_child_samples' : [10, 20, 30, 50],
    'feature_fraction'  : [0.6, 0.7, 0.8, 0.9, 1.0],
    'bagging_fraction'  : [0.6, 0.7, 0.8, 0.9, 1.0],
    'bagging_freq'      : [0, 1, 3, 5],
    'reg_alpha'         : [0, 0.1, 0.5, 1.0],
    'reg_lambda'        : [0, 0.1, 0.5, 1.0]
}

lgbm_base = LGBMRegressor(random_state=42, verbose=-1)

lgbm_search = RandomizedSearchCV(
    estimator  = lgbm_base,
    param_distributions = lgbm_param_dist,
    n_iter     = 50,
    cv         = kf,
    scoring    = 'r2',
    n_jobs     = -1,
    random_state = 42,
    verbose    = 1
)

lgbm_search.fit(X_train, y_train)
lgbm_model = lgbm_search.best_estimator_

print(f"\n🏆 Best Parameters LightGBM:")
for param, val in lgbm_search.best_params_.items():
    print(f"   {param:20s}: {val}")

lgbm_results, lgbm_pred = evaluate_model(
    lgbm_model, X_train, X_test, y_train, y_test,
    "LightGBM (Tuned)"
)

all_results['LightGBM'] = lgbm_results
all_predictions['LightGBM'] = lgbm_pred

joblib.dump(lgbm_model, 'models/lgbm_model.pkl')
print("\n✅ Model tersimpan: models/lgbm_model.pkl")

🚀 Tuning LightGBM dengan RandomizedSearchCV...
   (Estimasi waktu: 2-5 menit)
Fitting 5 folds for each of 50 candidates, totalling 250 fits

🏆 Best Parameters LightGBM:
   reg_lambda          : 0.1
   reg_alpha           : 1.0
   num_leaves          : 80
   n_estimators        : 200
   min_child_samples   : 10
   max_depth           : 5
   learning_rate       : 0.05
   feature_fraction    : 0.9
   bagging_freq        : 1
   bagging_fraction    : 0.9

📊 LightGBM (Tuned)
Metrik          Train       Test
--------------------------------
RMSE           0.1372     0.1560
MAE            0.1091     0.1257
R²             0.6738     0.6114

Overfitting Check: ✅ Normal (gap=0.0624)

✅ Model tersimpan: models/lgbm_model.pkl


Ridge Regression

In [6]:
# ── RIDGE REGRESSION ───────────────────────────────────────
from sklearn.model_selection import GridSearchCV

print("📐 Tuning Ridge Regression dengan GridSearchCV...")

ridge_param_grid = {
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 
              10.0, 50.0, 100.0, 500.0, 1000.0]
}

ridge_base = Ridge()

# GridSearchCV karena parameter sedikit → exhaustive lebih baik
ridge_search = GridSearchCV(
    estimator  = ridge_base,
    param_grid = ridge_param_grid,
    cv         = kf,
    scoring    = 'r2',
    n_jobs     = -1,
    verbose    = 0
)

ridge_search.fit(X_train, y_train)
ridge_model = ridge_search.best_estimator_

print(f"🏆 Best Alpha Ridge: {ridge_search.best_params_['alpha']}")

ridge_results, ridge_pred = evaluate_model(
    ridge_model, X_train, X_test, y_train, y_test,
    "Ridge Regression (Tuned)"
)

all_results['Ridge'] = ridge_results
all_predictions['Ridge'] = ridge_pred

joblib.dump(ridge_model, 'models/ridge_model.pkl')
print("\n✅ Model tersimpan: models/ridge_model.pkl")

📐 Tuning Ridge Regression dengan GridSearchCV...
🏆 Best Alpha Ridge: 0.1

📊 Ridge Regression (Tuned)
Metrik          Train       Test
--------------------------------
RMSE           0.1495     0.1531
MAE            0.1182     0.1229
R²             0.6129     0.6258

Overfitting Check: ✅ Normal (gap=-0.0129)

✅ Model tersimpan: models/ridge_model.pkl


Standard Vector Regression


In [7]:
# ── SVR ────────────────────────────────────────────────────
# Non-linear model, bagus untuk dataset kecil yang sudah di-scale

print("🎯 Tuning SVR dengan RandomizedSearchCV...")
print("   (Estimasi waktu: 3-7 menit, SVR paling lambat)")

svr_param_dist = {
    'C'      : [0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
    'epsilon': [0.01, 0.05, 0.1, 0.2, 0.5],
    'kernel' : ['rbf', 'linear', 'poly'],
    'gamma'  : ['scale', 'auto', 0.01, 0.1, 1.0]
}

svr_base = SVR()

svr_search = RandomizedSearchCV(
    estimator  = svr_base,
    param_distributions = svr_param_dist,
    n_iter     = 50,
    cv         = kf,
    scoring    = 'r2',
    n_jobs     = -1,
    random_state = 42,
    verbose    = 1
)

svr_search.fit(X_train, y_train)
svr_model = svr_search.best_estimator_

print(f"\n🏆 Best Parameters SVR:")
for param, val in svr_search.best_params_.items():
    print(f"   {param:10s}: {val}")

svr_results, svr_pred = evaluate_model(
    svr_model, X_train, X_test, y_train, y_test,
    "SVR (Tuned)"
)

all_results['SVR'] = svr_results
all_predictions['SVR'] = svr_pred

joblib.dump(svr_model, 'models/svr_model.pkl')
print("\n✅ Model tersimpan: models/svr_model.pkl")

🎯 Tuning SVR dengan RandomizedSearchCV...
   (Estimasi waktu: 3-7 menit, SVR paling lambat)
Fitting 5 folds for each of 50 candidates, totalling 250 fits

🏆 Best Parameters SVR:
   kernel    : rbf
   gamma     : auto
   epsilon   : 0.1
   C         : 10.0

📊 SVR (Tuned)
Metrik          Train       Test
--------------------------------
RMSE           0.1476     0.1527
MAE            0.1169     0.1229
R²             0.6229     0.6277

Overfitting Check: ✅ Normal (gap=-0.0048)

✅ Model tersimpan: models/svr_model.pkl


In [8]:
results_df = pd.DataFrame(all_results).T.reset_index(drop=True)
results_df = results_df.sort_values('R2_Test', ascending=False).reset_index(drop=True)
results_df.index += 1

print("="*95)
print("🏆 PERBANDINGAN SEMUA MODEL REGRESI")
print("="*95)
print(f"{'':30s} {'──── TRAIN ────':^20s} {'──────── TEST ────────':^30s}")
print(f"{'Model':30s} {'RMSE':>8s} {'R²':>8s} {'RMSE':>8s} {'MAE':>8s} {'R²':>8s} {'Overfit':>25s}")
print("-"*100)
for _, row in results_df.iterrows():
    print(f"{row['Model']:30s} "
          f"{float(row['RMSE_Train']):>8.4f} {float(row['R2_Train']):>8.4f} "
          f"{float(row['RMSE_Test']):>8.4f} {float(row['MAE_Test']):>8.4f} "
          f"{float(row['R2_Test']):>8.4f} {row['Overfit']:>25s}")

print(f"\n📌 Keterangan:")
print(f"   RMSE ↓ → makin kecil makin baik")
print(f"   MAE  ↓ → makin kecil makin baik")
print(f"   R²   ↑ → makin besar makin baik (max 1.0)")

best_model_name = results_df.iloc[0]['Model']
print(f"\n🥇 Model terbaik: {best_model_name}")
print(f"   R² Test  : {results_df.iloc[0]['R2_Test']}")
print(f"   RMSE Test: {results_df.iloc[0]['RMSE_Test']}")
print(f"   MAE Test : {results_df.iloc[0]['MAE_Test']}")

🏆 PERBANDINGAN SEMUA MODEL REGRESI
                                 ──── TRAIN ────        ──────── TEST ────────    
Model                              RMSE       R²     RMSE      MAE       R²                   Overfit
----------------------------------------------------------------------------------------------------
SVR (Tuned)                      0.1476   0.6229   0.1527   0.1229   0.6277    ✅ Normal (gap=-0.0048)
Ridge Regression (Tuned)         0.1495   0.6129   0.1531   0.1229   0.6258    ✅ Normal (gap=-0.0129)
XGBoost (Tuned)                  0.1408   0.6569   0.1556   0.1248   0.6135     ✅ Normal (gap=0.0434)
LightGBM (Tuned)                 0.1372   0.6738   0.1560   0.1257   0.6114     ✅ Normal (gap=0.0624)
Random Forest (Baseline)         0.0626   0.9321   0.1613   0.1290   0.5849   ⚠️ OVERFIT (gap=0.3472)

📌 Keterangan:
   RMSE ↓ → makin kecil makin baik
   MAE  ↓ → makin kecil makin baik
   R²   ↑ → makin besar makin baik (max 1.0)

🥇 Model terbaik: SVR (Tuned)
   R² Tes

In [9]:
# ── Set XGBoost sebagai best model ──────────────────────────
import joblib
import shap
import numpy as np

# Load data
X_train, X_test, y_train, y_test = joblib.load('models/data_split_regression.pkl')

# Load XGBoost model yang sudah di-tune
xgb_model = joblib.load('models/xgb_model.pkl')

# ── Feature Importance ───────────────────────────────────────
feature_cols = joblib.load('models/feature_cols.pkl')
importance = xgb_model.feature_importances_

import pandas as pd
feat_imp = pd.Series(importance, index=feature_cols).sort_values(ascending=False)
print("Feature Importance XGBoost:")
print(feat_imp)

# ── SHAP TreeExplainer ───────────────────────────────────────
print("\nMenghitung SHAP values...")
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_test)
print("✅ SHAP values selesai!")

# ── Simpan untuk website ─────────────────────────────────────
joblib.dump(xgb_model,        'models/best_regression_model.pkl')  # override best model
joblib.dump(explainer_xgb,    'models/shap_explainer_xgb.pkl')
joblib.dump(shap_values_xgb,  'models/shap_values_xgb.pkl')
joblib.dump(feat_imp,         'models/xgb_feature_importance.pkl')

print("✅ Semua tersimpan!")
print("   models/best_regression_model.pkl → XGBoost")
print("   models/shap_explainer_xgb.pkl")
print("   models/shap_values_xgb.pkl")
print("   models/xgb_feature_importance.pkl")

Feature Importance XGBoost:
platform_Agoda                  0.300145
facility                        0.196433
cleanliness                     0.131616
service                         0.104481
location                        0.077624
is_facility_available           0.056565
platform_Tiket.com              0.033101
value_for_money                 0.029461
platform_Trip.com               0.022660
platform_Booking.com            0.018157
platform_Traveloka              0.015829
is_value_for_money_available    0.013928
dtype: float32

Menghitung SHAP values...
✅ SHAP values selesai!
✅ Semua tersimpan!
   models/best_regression_model.pkl → XGBoost
   models/shap_explainer_xgb.pkl
   models/shap_values_xgb.pkl
   models/xgb_feature_importance.pkl
